In [ ]:
import pandas as pd
import json
import numpy as np
np.random.seed(42)

cats = ["bottle", "bowl", "chair", "cup", 
                "door", "spoon", "table", "window"]

subjs = ["030FD", "031HW", "032SR", "033SE", "036MR", "037SM", "038RB", "039ED", "041EC",
          "042MB", "043MP", "045NA", "046TE", "047MS"]

In [ ]:
# Open and load the JSON file
with open("../instance_filenames.json", "r") as f:
    top_nonTop_instances = json.load(f) # this is just a list of filenames that belong to Rank 1 or Other instances (or both) --- just used as a sanity check below

df_sub_data = pd.read_csv(f"../../data/PanelA/sub_data_release.csv")

# clean up
cols = list(df_sub_data.columns)
for col in cols:
    try:
        df_sub_data[col] = df_sub_data[col].str.strip(' "\'')
    except:
        continue

df_rank_data = pd.read_csv(f"../../data/PanelA/rank_data_release.csv")

# clean up
cols = list(df_rank_data.columns)
for col in cols:
    try:
        df_rank_data[col] = df_rank_data[col].str.strip(' "\'')
    except:
        continue

In [ ]:
print(df_sub_data)
print(df_rank_data)

In [ ]:
# this collects all imgs completely clean filtered
# but it doesnt sample uniformly yet

# collect all dfs here
dfs_bySubj_andCat = []

for cat in cats:
    # get all rows that contain these imgs
    df_sub_data_filtered = df_sub_data[ df_sub_data['obj'] == cat ].copy()
    df_rank_data_filtered = df_rank_data[ df_rank_data['obj'] == cat ].copy()

    # get subjs in this cat (some subjs dont have some categories)
    subjs_sub = list( dict( df_sub_data_filtered['subj'].value_counts() ).keys() )
    subjs_rank = list( dict( df_rank_data_filtered['subj'].value_counts() ).keys() )
    
    if set(subjs_sub) != set(subjs_rank):
        print(subjs_sub)
        print(subjs_rank)
    assert set(subjs_sub) == set(subjs_rank)
    # print(df_rank_data_filtered.columns) ['subj', 'obj_id', 'n', 'rank', 'obj']
    # get rank 1 instance for each subj 
    counts = []
    for subj in subjs_sub:
        
        # get all instances with at least 5 tokens
        instance_list = df_rank_data_filtered[
            (df_rank_data_filtered['subj'] == subj) & \
            (df_rank_data_filtered['obj'] == cat) & \
            (df_rank_data_filtered['n'] >= 5)
        ]['obj_id'].values

        # n = df_rank_data_filtered[ (df_rank_data_filtered['subj'] == subj) & (df_rank_data_filtered['obj'] == cat) ]#['n']
        # print(n)

        # print(instance_list) # ['e' 'a' 'd' 'h' 'b' 'c' 'f' 'g' 'i' 'j']
        # print(n)

        # if none move on
        if len(instance_list) == 0:
            continue
        
        # the count (n) above counts uncodable imgs for similarity
        # so now we run the same thing again but only keeping acceptable imgs
        obj_id_counts = {}
        for obj_id in instance_list:
            final_df = df_sub_data_filtered[
                (df_sub_data_filtered['subj'] == subj) & \
                (df_sub_data_filtered['obj'] == cat) & \
                (df_sub_data_filtered['obj_id'] == obj_id) & \
                (df_sub_data_filtered['frame_id'].isin(top_nonTop_instances[subj]['all'][cat])) & \
                (df_sub_data_filtered['frame_id'].isin(top_nonTop_instances[subj]['acceptable_imgs'][cat]))
            ].copy()
            
            obj_id_counts[obj_id] = final_df['frame_id'].nunique()

            del final_df
        
        # print(obj_id_counts) # {'e': 89, 'o': 26, 'p': 23, 'i': 11, 'h': 10, 'j': 8, 'l': 10, 'n': 6, 'k': 4}

        # then we filter again instances with less than 5 imgs
        filtered_obj = {k: v for k, v in obj_id_counts.items() if v >= 5}
        if len(filtered_obj) == 0:
            continue
        # print(filtered_obj) # {'e': 89, 'o': 26, 'p': 23, 'i': 11, 'h': 10, 'j': 8, 'l': 10, 'n': 6}

        min_imgs = min ( list( filtered_obj.values() ) )
        
        exact_uniform_per_id_per_subj = []
        # sample (exactly) uniformly from each instance for this subj and cat
        for k, v in filtered_obj.items():
            assert v >= 5
            assert k in instance_list

            # get all imgs for these obj_ids that passed the filtering above again
            final_df = df_sub_data_filtered[
                (df_sub_data_filtered['subj'] == subj) & \
                (df_sub_data_filtered['obj'] == cat) & \
                (df_sub_data_filtered['obj_id'] == k) & \
                (df_sub_data_filtered['frame_id'].isin(top_nonTop_instances[subj]['all'][cat])) & \
                (df_sub_data_filtered['frame_id'].isin(top_nonTop_instances[subj]['acceptable_imgs'][cat]))
            ].copy()
            
            assert final_df['frame_id'].nunique() == obj_id_counts[k]

            df_sampled = final_df.sample(n=min_imgs, random_state=42, replace=False)
            exact_uniform_per_id_per_subj.append(df_sampled)
            
            del final_df, df_sampled

        df_exact_uniform_per_id_per_subj = pd.concat(exact_uniform_per_id_per_subj, ignore_index=True)
        
        # if less than 10 imgs overall (across all IDs) move on
        if df_exact_uniform_per_id_per_subj['frame_id'].nunique() < 10:
            continue
        
        # record number of imgs get which subj has the least data per cat
        counts.append(df_exact_uniform_per_id_per_subj['frame_id'].nunique())

        # print(df_sub_data_filtered['frame_id'].values)
        # print(top_nonTop_instances[subj]['acceptable_imgs'][cat])
        
        # collect dfs
        dfs_bySubj_andCat.append(df_exact_uniform_per_id_per_subj)

        del df_exact_uniform_per_id_per_subj, instance_list, obj_id_counts, filtered_obj
    
    print(f"Cat {cat} min: {min(counts)}")
    
    del df_sub_data_filtered, df_rank_data_filtered, subjs_sub, subjs_rank, counts

# all cats and subjs together
to_file_df = pd.concat(dfs_bySubj_andCat, ignore_index=True)

# print(counts) output:
    # Cat bottle min: 11
    # Cat bowl min: 10
    # Cat chair min: 19
    # Cat cup min: 10
    # Cat door min: 15
    # Cat spoon min: 12
    # Cat table min: 12
    # Cat window min: 10

In [ ]:
to_file_df.to_csv("../../PanelA/RGB_hist/NULL_beforeSampling.csv", index=False)

In [ ]:
samples_perCat = { # min subj counts from before
    "bottle": 11,
    "bowl": 10,
    "chair": 19,
    "cup": 10,
    "door": 15,
    "spoon": 12,
    "table": 12,
    "window": 10,
}

In [ ]:
# this now samples uniformly

df_clean = pd.read_csv("../../PanelA/RGB_hist/NULL_beforeSampling.csv")
# collect all dfs here
dfs_bySubj_andCat = []

for cat in cats:
    # get all rows that contain these imgs
    df_sub_data_filtered = df_sub_data[ df_sub_data['obj'] == cat ].copy()
    df_rank_data_filtered = df_rank_data[ df_rank_data['obj'] == cat ].copy()

    # get subjs in this cat (some subjs dont have some categories)
    subjs_sub = list( dict( df_sub_data_filtered['subj'].value_counts() ).keys() )
    subjs_rank = list( dict( df_rank_data_filtered['subj'].value_counts() ).keys() )
    
    if set(subjs_sub) != set(subjs_rank):
        print(subjs_sub)
        print(subjs_rank)
    assert set(subjs_sub) == set(subjs_rank)
    # print(df_rank_data_filtered.columns) ['subj', 'obj_id', 'n', 'rank', 'obj']
    # get rank 1 instance for each subj 

    for subj in subjs_sub:

        # for this subj and cat
        df_tosample = df_clean [ 
            (df_clean['obj'] == cat) & \
            (df_clean['subj'] == subj)
        ].copy()
        
        if df_tosample.shape[0] == 0:
            continue
        # remove dup frames
        df_tosample.drop_duplicates(subset=["frame_id"], inplace=True)
        
        # if less than 10 imgs move on
        if df_tosample['frame_id'].nunique() < 10:
            print(df_tosample.shape)
            raise
            # continue

        df_sampled = df_tosample.sample(n=samples_perCat[cat], random_state=42, replace=False)
        dfs_bySubj_andCat.append(df_sampled)
        
        del df_sampled, df_tosample
    
    del df_sub_data_filtered, df_rank_data_filtered, subjs_sub, subjs_rank

to_file_df = pd.concat(dfs_bySubj_andCat, ignore_index=True)
    

In [ ]:
to_file_df.to_csv("../../PanelA/RGB_hist/NULL_afterSampling.csv", index=False)